In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, year, lit
from pyspark.sql.types import IntegerType, DateType

# 1. Create Spark session
spark = SparkSession.builder \
    .appName("SoccerDataPreparation") \
    .getOrCreate()

# 2. Load data from CSV file
# Note: Make sure the file path is correct inside the container
input_path = "../data/rim_championnat_results_2007-2025.csv"
df = spark.read.csv(input_path, header=True, inferSchema=True)

print("Original Schema:")
df.printSchema()
df.show(5)

# 3. Data Cleaning
# a- Convert columns to their correct types (especially goals and dates)
# b- Handle missing values (we will drop rows with missing values in essential columns)
df_clean = df.dropna(subset=["date", "home_team", "away_team", "home_goals", "away_goals"]) \
             .withColumn("home_goals", col("home_goals").cast(IntegerType())) \
             .withColumn("away_goals", col("away_goals").cast(IntegerType())) \
             .wiSthColumn("date", col("date").cast(DateType()))

# 4. Create derived columns - as required in the project
df_enriched = df_clean \
    .withColumn("total_goals", col("home_goals") + col("away_goals")) \
    .withColumn("goal_difference", col("home_goals") - col("away_goals")) \
    .withColumn("result", 
        when(col("home_goals") > col("away_goals"), "home_win")
        .when(col("home_goals") < col("away_goals"), "away_win")
        .otherwise("draw")) \
    .withColumn("match_year", year(col("date")))

# 5. Show final results for verification
print("Enriched Schema:")
df_enriched.printSchema()
df_enriched.show(5)

# 6. Save data in Parquet format
output_path = "outputs/prepared/cleaned_soccer_data.parquet"
df_enriched.write.mode("overwrite").parquet(output_path)

print(f"Success! Data saved to {output_path}")S